# Class 2 lab: binary classification, yes or no

**CML3112, Civil Engineering and Architecture**  |  Week 9  |  KLAi at IUEA

In this lab the model answers one question: is it **high cooling load**? This is **binary classification**: there are two classes. Yes is the **positive class** (1) and no is the **negative class** (0). The actual class is **positive** when the cooling load is 54 kW or more, and **negative** when it is below. You build two models, count their right and wrong answers, and move the decision threshold. You also check the formulas from the slides against the printed numbers.

**How to use this notebook**

1. Run the cells in order from the top. Press Shift and Enter to run a cell.
2. Every step has the same rhythm: **PREDICT**, **RUN**, **READ**, then **CHANGE** one thing and see what moves.
3. Type your answers in the lines that start with `>`. Double-click a text cell to edit it.
4. If you get an error, read the last line first. The slide "Four messages you will meet" lists the common ones.

**The words** (also on the slides): the *label* is the target variable of a classification, here 1 (positive, yes) or 0 (negative, no), made from the target with the limit. *Imputation* means filling a missing value (NaN), and `SimpleImputer` is the scikit-learn tool for it. The *decision threshold* is the probability at or above which the model predicts positive (yes). The *confusion matrix* is a table of four counts: *true positives* (TP: predicted positive and actually positive), *false negatives* (FN: predicted negative but actually positive), *false positives* (FP: predicted positive but actually negative) and *true negatives* (TN: predicted negative and actually negative). *Precision* is the share of the positive predictions that are really positive. *Recall* (also called *sensitivity* or *true positive rate*) is the share of the actual positives that the model finds. The *ROC curve* plots the true positive rate against the false positive rate, and *AUC* (area under the curve) says how well the model ranks actual positive rows above actual negative rows.

---
## Step 1: Prepare the data (the same as Class 1, shorter)

Upload `engineering_data.csv` in the Files panel on the left of Colab if you have not already. We clean a copy, split by equipment group into a training set and a validation set, and impute the one missing value (NaN) with the median of its column, taken from the training set only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# read the file. The raw table is never changed: we clean a copy.
raw_data = pd.read_csv("engineering_data.csv")

modelling_data = raw_data.drop_duplicates("record_id").copy()                        # keep the first copy of each record_id
has_target = modelling_data["target"].notna()                                         # True where the target is recorded
load_ok = modelling_data["load_pct"].between(0, 100)                                  # True where load is between 0 and 100
modelling_data = modelling_data[has_target & load_ok].copy()
print("rows left to model with:", len(modelling_data))

# split by equipment group: E1 to E4 are the training set, E5 the validation set, E6 stays sealed as the test set (we do not touch it)
training_data = modelling_data[modelling_data["equipment_id"].isin(["E1", "E2", "E3", "E4"])].copy()
validation_data = modelling_data[modelling_data["equipment_id"] == "E5"].copy()
print("training, validation, test rows:", len(training_data), len(validation_data), len(modelling_data[modelling_data["equipment_id"] == "E6"]))

In [ ]:
from sklearn.impute import SimpleImputer

feature_columns = ["load_pct", "ambient_c"]

# imputation: the median of each column is learned from the training set only, then used to fill the missing value in both tables
imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(training_data[feature_columns])
X_valid = imputer.transform(validation_data[feature_columns])
print("missing values left:", int(np.isnan(X_train).sum() + np.isnan(X_valid).sum()))

**READ the output.** Do the row counts match the numbers you saw in Class 1? Why do we impute the missing value with a median learned from the training set only?

> My answer: 

---
## Step 2: Make the label and meet the majority-class baseline

The label is the target variable of a classification. It is **positive (1, yes)** when the cooling load is at or above the limit, and **negative (0, no)** below it. A **majority-class baseline** is the simplest possible classifier: it always predicts the most common class, which here is no. Every real model has to beat it.

**PREDICT first.** Out of the 15 validation rows, how many do you think have a target at or above the limit (positive)? What accuracy would the majority-class baseline get by answering no for every row?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
limit = 54   # kW, the cohort limit for "high cooling load"

y_train = (training_data["target"] >= limit).astype(int).to_numpy()     # 1 = positive (yes), 0 = negative (no)
y_valid = (validation_data["target"] >= limit).astype(int).to_numpy()

print("positive labels in training:", int(y_train.sum()), "of", len(y_train))
print("positive labels in validation:", int(y_valid.sum()), "of", len(y_valid))

# the majority-class baseline: predict the most common class (no) for every row
baseline_pred = np.zeros(len(y_valid), dtype=int)
baseline_accuracy = float(np.mean(baseline_pred == y_valid))
print(f"Accuracy of the majority-class baseline (always no): {baseline_accuracy:.2f}")

**READ the output.** Answering no for every row gives a high accuracy, yet it finds no actual positive case. How many of the actual positive rows does it find, and why does this make accuracy alone a poor score when one class is much rarer than the other (class imbalance)?

> My answer: 

---
## Step 3: Model 1: standardisation and logistic regression

Standardisation (z-score scaling, done by `StandardScaler`) rewrites each feature value as the number of standard deviations it lies above or below the training mean: x' = (x - mean) / standard deviation. The standard deviation is the typical distance of a value from the mean. Logistic regression then builds a score s from the standardised features and passes it through the sigmoid (logistic) function, which turns it into a probability between 0 and 1. It learns its weights by making the **log loss** (binary cross-entropy) small: the loss of one row is -(y ln p + (1 - y) ln(1 - p)), and the training loss is the mean over the training rows.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)     # mean and standard deviation from the training set only
print(f"training mean of load_pct: {scaler.mean_[0]:.2f}   standard deviation: {scaler.scale_[0]:.2f}")
print(f"training mean of ambient_c: {scaler.mean_[1]:.2f}   standard deviation: {scaler.scale_[1]:.2f}")
first_row = scaler.transform(X_train[:1])[0]
print(f"first training row, standardised: {first_row[0]:.2f} {first_row[1]:.2f}")

**READ the output.** Check the first standardised load by hand. Print `X_train[0]`. It shows two numbers, and the first number is the load_pct of the first training row. Then work out (that load minus the mean) divided by the standard deviation, using the mean and standard deviation printed above. Does your answer match the first standardised number? (StandardScaler divides by n, as NumPy `X_train.std(axis=0)` does. The pandas method `.std()` divides by n - 1, so it gives a slightly larger number.)

> My answer: 

**PREDICT first.** Will the weight (coefficient) for load_pct be positive or negative? What about ambient_c? A positive weight means that more of that feature pushes the prediction toward positive (yes).

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = make_pipeline(StandardScaler(), LogisticRegression())
logistic_model.fit(X_train, y_train)                              # training: learn mean, standard deviation and weights from the training set
y_proba = logistic_model.predict_proba(X_valid)[:, 1]             # predicted probability of the positive class for each validation row
threshold = 0.5                                                   # the decision threshold
y_pred = (y_proba >= threshold).astype(int)                       # at or above the threshold: predict positive (1), otherwise negative (0)

learned = logistic_model.named_steps["logisticregression"]
print(f"weight for load_pct: {learned.coef_[0][0]:.2f}")
print(f"weight for ambient_c: {learned.coef_[0][1]:.2f}")
print(f"intercept: {learned.intercept_[0]:.2f}")
print("first 5 probabilities:", " ".join(f"{p:.2f}" for p in y_proba[:5]))
print("first 5 actual classes: ", y_valid[:5])

**READ the output.** Both features were standardised, so the two weights can be compared. Which feature has the larger weight? A larger weight means that one standard deviation more of that feature raises the score more.

> My answer: 

**PREDICT first.** The training loss starts at 0.69 (every probability 0.5) before training. Will the mean log loss of the trained model on the training set be closer to 0 or to 0.69?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
p_train = logistic_model.predict_proba(X_train)[:, 1]                               # probability of the positive class for each training row
row_loss = -(y_train * np.log(p_train) + (1 - y_train) * np.log(1 - p_train))     # log loss of each training row
print(f"mean log loss on the training set: {row_loss.mean():.2f}")

**READ the output.** This is the number that training makes small. scikit-learn minimises it with its solver (lbfgs), plus a small L2 penalty on the size of the weights (C = 1.0 by default).

> My answer: 

**CHANGE one thing.** Remove the standardisation. Run the cell below: it fits `plain_model = LogisticRegression(max_iter=1000)` on the raw `X_train` and prints its weights, its confusion matrix at threshold 0.5 and the largest change in a validation probability. Compare each weight with the standardised weight above: which changed most? Did the confusion matrix change? The reason is the L2 penalty: LogisticRegression adds the squared weights to the loss, in whatever units the features have. One percent of load and one degree of ambient temperature are treated alike, so the fitted model is different. That is one more reason to standardise first.

> What I saw: 

In [ ]:
# your experiment: no StandardScaler
from sklearn.metrics import confusion_matrix

plain_model = LogisticRegression(max_iter=1000)
plain_model.fit(X_train, y_train)
print("weights without standardisation:", np.round(plain_model.coef_[0], 3))
plain_proba = plain_model.predict_proba(X_valid)[:, 1]
print("confusion matrix without standardisation:", confusion_matrix(y_valid, (plain_proba >= 0.5).astype(int)).tolist())
print(f"largest change in a validation probability: {np.max(np.abs(plain_proba - y_proba)):.2f}")

---
## Step 4: Model 2: a decision tree classifier

A row follows the yes or no questions (internal nodes, each a split on a feature and a threshold) down the tree until it reaches one leaf node at the bottom. That leaf node predicts positive when most of the rows of the training set that ended in it are positive, and negative otherwise. A tree needs no standardisation. `max_depth=3` is a hyperparameter (a setting chosen before training): the largest number of questions a row can be asked on the way down. `random_state=17` fixes the random choices, so everyone gets the same tree.

**PREDICT first.** Will the tree find the same actual positive rows as the logistic regression? Write down your guess before you run the cell.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=3, random_state=17)
tree_model.fit(X_train, y_train)
tree_y_pred = tree_model.predict(X_valid)
print("first 5 tree predictions:", tree_y_pred[:5])

**READ the output.** The tree predicts a class (0 or 1) directly. Step 5 counts how many of its answers are right. On the slide, the numbers in brackets in each leaf node show how many training rows a rule rests on.

> My answer: 

---
## Step 5: Count the answers: confusion matrix and four scores

In the confusion matrix that scikit-learn prints, the rows are the **Actual** class (negative, then positive) and the columns are the **Predicted** class (negative, then positive). So the matrix reads [[TN, FP], [FN, TP]]. The slides draw it in the same way.

**PREDICT first.** Before you run the cell, guess how many true positives (TP) the logistic model gets, and how many false positives (FP) it gives.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def evaluate_classifier(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)                    # rows: Actual (negative, positive). Columns: Predicted (negative, positive)
    tn, fp, fn, tp = cm.ravel()                              # order: TN, FP, FN, TP
    print(name)
    print("  confusion matrix, Actual rows and Predicted columns:", cm.tolist())
    print("  TP (true positives):", tp)
    print("  FN (false negatives):", fn)
    print("  FP (false positives):", fp)
    print("  TN (true negatives):", tn)
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)      # zero_division=0: if the model never predicts positive, precision is shown as 0
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"  {name}: accuracy {accuracy:.2f}, precision {precision:.2f}, recall {recall:.2f}, F1 {f1:.2f}")

evaluate_classifier("Logistic regression", y_valid, y_pred)
evaluate_classifier("Decision tree", y_valid, tree_y_pred)

**READ the output.** Compare the accuracy with the accuracy of the majority-class baseline from Step 2. Which score tells you more about how well the model finds the actual positive rows, and why?

> My answer: 

**Check by hand.** Take the four counts the logistic model printed. Work out the scores on paper and compare them with the printed scores:

- accuracy = (TP + TN) / (TP + TN + FP + FN)
- precision = TP / (TP + FP)
- recall (sensitivity) = TP / (TP + FN)
- F1 = 2 x precision x recall / (precision + recall)

> My arithmetic: 

---
## Step 6: Move the decision threshold and choose it with a cost table

The model gives a probability for each row. The decision threshold decides how high the probability must be before the prediction is positive. Moving it changes the precision-recall trade-off. To choose, we put a price on each mistake: **cost = MISSED_COST x FN + FALSE_ALARM_COST x FP**. The costs 5 and 1 are invented teaching numbers: one missed positive costs as much as five false alarms. The rule: choose the threshold with the lowest cost on the validation set; if two tie, take the higher threshold.

**PREDICT first.** If you lower the decision threshold from 0.5 to 0.3, will recall go up or down? What will happen to the false positives (FP)? Which of 0.3, 0.5 and 0.7 do you expect to have the lowest cost?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
MISSED_COST = 5          # invented teaching cost of one false negative (a missed positive)
FALSE_ALARM_COST = 1     # invented teaching cost of one false positive (a false alarm)

cost_table = []
for threshold in [0.3, 0.5, 0.7]:
    y_pred_t = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_pred_t).ravel()
    precision = precision_score(y_valid, y_pred_t, zero_division=0)
    recall = recall_score(y_valid, y_pred_t, zero_division=0)
    cost = MISSED_COST * fn + FALSE_ALARM_COST * fp
    cost_table.append({"threshold": threshold, "cost": int(cost)})
    print(f"threshold {threshold}: TP {tp}, FP {fp}, FN {fn}, TN {tn}, recall {recall:.2f}, precision {precision:.2f}, cost {cost}")

# lowest cost first; if two tie, the higher threshold comes first
best = sorted(cost_table, key=lambda row: (row["cost"], -row["threshold"]))[0]
print(f"chosen threshold: {best['threshold']} (validation cost {best['cost']})")

**READ the output.** Check one cost by hand: 5 x FN + 1 x FP. Is the chosen threshold the one you would pick for the building? Give one sentence of reasoning.

> My answer: 

**CHANGE one thing.** Change the list `[0.3, 0.5, 0.7]` to `[0.1, 0.2, 0.3, 0.5, 0.7]` and run the cell again. At which thresholds do false positives appear? Does recall ever fall as the threshold goes down? Does the chosen threshold change?

> What I saw: 

---
## Step 7: The ROC curve and AUC

The ROC curve (receiver operating characteristic) plots the true positive rate (TPR, the same as recall or sensitivity) against the false positive rate (FPR, the share of actual negatives wrongly predicted positive), one point for every decision threshold. AUC is the area under the curve: how well the model ranks actual positive rows above actual negative rows. With only 15 rows there are only a few points, so the line is a staircase. With thousands of rows it would look smooth.

**PREDICT first.** All three actual positive rows had higher probabilities than most negatives. Will the AUC be above 0.9?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_valid, y_proba)
auc = roc_auc_score(y_valid, y_proba)
print(f"AUC: {auc:.2f}")

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")                      # the line for random guessing, AUC 0.5
plt.xlabel("false positive rate (FPR)")
plt.ylabel("true positive rate (TPR, recall)")
plt.title("ROC curve, validation set")
plt.show()

**READ the output.** The dashed line is random guessing, with AUC 0.5. How far above the dashed line does your curve run? Your plot shows only a few points, because `roc_curve` keeps only the corners of the staircase (the slide draws every point). The validation set has only 15 rows and 3 actual positives: how much would you trust this AUC?

> My answer: 

---
## Step 8: Your conclusion

Use the numbers you printed.

**Write your conclusion in three sentences.** Which model found more actual positives? Which decision threshold would you use, and why? What is the biggest weakness of the evidence?

> My conclusion: 

**What this does not prove.** The data are synthetic teaching data. The validation set has only 15 rows and 3 positive cases, so one different prediction changes recall a lot. The test set (E6) has not been used. In Class 3 we learn how to judge a model with more care.

---
## Step 9: Save your work

Choose **Runtime, then Restart session and run all**. If every cell runs without an error, choose **File, then Save a copy in Drive**. This notebook is for learning. Part B of the Block 1 assignment (A8, classification and cost) repeats these steps in the assignment notebook. Block work is practice and does not count toward the final mark.